# 08.5 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/05-first-keras-network/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow start-up messages
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import keras
from keras import layers
keras.utils.set_random_seed(42)      # same starting weights every run (small differences across machines are normal)
print("Keras", keras.__version__, "on", keras.backend.backend())

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]
cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)            # fit on training customers only (lesson 04.5)
Xs_train, Xs_test = scaler.transform(X_train), scaler.transform(X_test)

def churn_network(hidden=(16, 8), seed=42):
    keras.utils.set_random_seed(seed)                  # seed decides the starting weights
    model = keras.Sequential([keras.Input((5,))] +
                             [layers.Dense(units, activation="relu") for units in hidden] +
                             [layers.Dense(1, activation="sigmoid")])
    model.compile(optimizer="adam", loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.AUC(name="auc")])
    return model

from sklearn.metrics import roc_auc_score
import math, time

Keras 3.15.1 on tensorflow


3,317 customers, churn rate 43.6%, train 2,487 / test 830


## Exercise 1

$5 \times 64 + 64 = 384$; $64 \times 64 + 64 = 4{,}160$; $64 + 1 = 65$. Total **4,609**, nineteen times the 5-16-8-1 network.

In [2]:
print(f"{churn_network((64, 64)).count_params():,} parameters")

4,609 parameters


## Exercise 2

In [3]:
n_fit = math.floor(len(Xs_train) * 0.8)
for bs in [8, 32, 256]:
    m = churn_network()
    start = time.time()
    m.fit(Xs_train, y_train, epochs=50, batch_size=bs, validation_split=0.2, verbose=0)
    auc = roc_auc_score(y_test, m.predict(Xs_test, verbose=0).ravel())
    print(f"batch {bs:3d}: {math.ceil(n_fit / bs):3d} steps/epoch, {time.time() - start:4.1f} s, test AUC {auc:.3f}")

batch   8: 249 steps/epoch, 46.9 s, test AUC 0.739


batch  32:  63 steps/epoch,  2.6 s, test AUC 0.740


batch 256:   8 steps/epoch,  1.7 s, test AUC 0.740


Small batches mean many more update steps (249 per epoch with 8 rows, 8 with 256) and so more time, but each step is noisier. Large batches are fast per epoch but make few updates, so they often need more epochs. On this small table every choice ends up near the same AUC; lesson 08.6 looks closer.

## Exercise 3

In [4]:
big = churn_network((64, 64))
big.fit(Xs_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)
print(f"5-64-64-1 test AUC (seed 42): {roc_auc_score(y_test, big.predict(Xs_test, verbose=0).ravel()):.3f}")

5-64-64-1 test AUC (seed 42): 0.745


That single run lands a little above 0.741. Before celebrating, rerun both networks with five different random seeds:

In [5]:
def seed_aucs(hidden, seeds=range(5)):
    aucs = []
    for seed in seeds:
        m = churn_network(hidden, seed=seed)
        m.fit(Xs_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)
        aucs.append(roc_auc_score(y_test, m.predict(Xs_test, verbose=0).ravel()))
    return np.array(aucs)

for hidden in [(16, 8), (64, 64)]:
    a = seed_aucs(hidden)
    print(f"5-{hidden[0]}-{hidden[1]}-1: AUC {a.round(3)}  mean {a.mean():.3f}")

5-16-8-1: AUC [0.737 0.741 0.741 0.744 0.745]  mean 0.742


5-64-64-1: AUC [0.74  0.736 0.738 0.739 0.736]  mean 0.738


Across seeds the big network is no better, if anything slightly worse on average: the single-run gain was luck in the starting weights. More parameters cannot invent information the five columns do not contain, and one run is never enough to claim a difference of a few thousandths (README section 8).

## Exercise 4

In [6]:
from sklearn.metrics import precision_score, recall_score
m = churn_network()
m.fit(Xs_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)
flag = (m.predict(Xs_test, verbose=0).ravel() > 0.5).astype(int)
print(f"flagged {flag.sum()} of {len(flag)} customers")
print(f"precision {precision_score(y_test, flag):.2f}, recall {recall_score(y_test, flag):.2f}")

flagged 363 of 830 customers
precision 0.63, recall 0.63


The same trade-off as lesson 03.7: the network's probabilities can be thresholded and costed exactly like logistic regression's. A neural network is just another probability-producing model to the business.